Connected to reactive-agent (3.11.x) (Python 3.11.-1)

 # Output guard

 **Last node before the response reaches the client** — two checks at different severities.

 - **Sensitive data — hard block:** 16-digit card numbers, API key patterns,
   `password: value` formats. Match → response blocked, never forwarded.
 - **Hallucination signals — soft flag:** phrases like "as of my training"
   or "I believe the price" are logged and surfaced as `hallucination_warning: True`
   in the return dict so the frontend can render a disclaimer badge.

In [ ]:
import re
from app.agent.state import AgentState
from app.core.logging import get_logger
from langchain_core.messages import AIMessage
import time

log = get_logger(__name__)

 ## Patterns

 `HALLUCINATION_SIGNALS` — soft flag, response still goes through.
 Blocking on hedging language would be too aggressive — sometimes it's appropriate.

 `SENSITIVE_PATTERNS` — hard block. The LLM leaked something it shouldn't have.

 Both use `re.IGNORECASE` except `SENSITIVE_PATTERNS` — card numbers and API keys
 are case-sensitive by nature.

In [ ]:
HALLUCINATION_SIGNALS = [re.compile(p, re.IGNORECASE) for p in [
    r"as of (my|my knowledge|training)",
    r"i (believe|think|assume) (that )?the (price|date|value)",
    r"approximately \$[\d,]+",
]]

SENSITIVE_PATTERNS = [re.compile(p) for p in [
    r"\b\d{16}\b",
    r"\b[A-Z0-9]{20,}\b",
    r"password\s*[:=]\s*\S+",
]]

 ## `output_validation_node`

 **Gap fixed — type check on last message:**
 Both checks now run on the last `AIMessage` specifically, not just the last message.
 If the last message is a `ToolMessage`, checking it would scan tool output
 instead of the agent's response — wrong target.

 **Gap fixed — hallucination flag surfaced to state:**
 `hallucination_warning: True` is now included in the return dict
 so the frontend can render a disclaimer badge instead of only reading logs.

 **Order fixed — empty check moved to top:**
 No point running regex on an empty string.

In [ ]:
async def output_validation_node(state: AgentState) -> dict:
    messages = state.get("messages", [])
    if not messages:
        return {"output_validated": False}

    # Fix: target the last AIMessage, not just the last message
    last_ai = next((m for m in reversed(messages) if isinstance(m, AIMessage)), None)
    if not last_ai:
        return {"output_validated": False}

    content = last_ai.content if hasattr(last_ai, "content") else ""

    # Fix: empty check first — no point scanning an empty string
    if not content.strip():
        return {"output_validated": False}

    hallucination_warning = False
    for pattern in HALLUCINATION_SIGNALS:
        if pattern.search(content):
            log.warning("hallucination_signal_detected: pattern=%s", pattern.pattern)
            hallucination_warning = True  # fix: surfaced to state, not just logged

    for pattern in SENSITIVE_PATTERNS:
        if pattern.search(content):
            log.error("sensitive_data_in_output: pattern=%s", pattern.pattern)
            return {
                "output_validated": False,
                "block_reason": "Sensitive data detected in output",
            }

    workflow_trace = state.get("workflow_trace", [])
    workflow_trace.append({
        "node": "output_validation",
        "status": "passed",
        "timestamp": time.time(),
    })

    return {
        "output_validated": True,
        "final_response": content,
        "hallucination_warning": hallucination_warning,
        "workflow_trace": workflow_trace,
    }